# 06. Multi-Channel Fusion: BM25 + Meta + OCR + ASR (`meme-organizer-fusion`)

In [1]:
# Cell 1: Imports, paths, and dataset loading
import json
from pathlib import Path

import cv2
import mlflow
import numpy as np
import optuna
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from torch import nn

from meme_organizer.config import CYRILLIC_OCR_DICT_PATH, CYRILLIC_OCR_MODEL_PATH, MLFLOW_EXPERIMENT_FUSION
from meme_organizer.data.dataset_loader import DatasetLoader
from meme_organizer.domain.ranking_metrics import RankingMetrics
from meme_organizer.evaluation.metrics.experiment_tracker import ExperimentTracker
from meme_organizer.evaluation.pipeline.evaluation_dataset_preparer import EvaluationDatasetPreparer
from meme_organizer.extraction.asr import SherpaOnnxAsrExtractor
from meme_organizer.extraction.audio import AudioExtractor
from meme_organizer.extraction.ocr import RapidOcrExtractor
from meme_organizer.extraction.video import KeyframeSampler
from meme_organizer.retrieval.fusion.gated_attention_fusion import GatedAttentionFusion
from meme_organizer.retrieval.fusion.multi_channel_scorer import MultiChannelScorer
from meme_organizer.retrieval.text.bm25_retriever import BM25Retriever
from meme_organizer.retrieval.text.fine_tuned_dense_retriever import FineTunedDenseRetriever

optuna.logging.set_verbosity(optuna.logging.WARNING)
ROOT = Path("..").resolve()
GOLD_PATH = ROOT / "data" / "gold" / "gold_dataset.jsonl"
SPLITS_PATH = ROOT / "data" / "splits" / "splits_v1.json"
SILVER_DIR = ROOT / "data" / "silver"
CACHE_PATH = ROOT / ".cache" / "ocr_asr_318.json"

corpus_docs = DatasetLoader.load_corpus(GOLD_PATH)
corpus_hashes = [d["file_hash"] for d in corpus_docs]
hash_to_idx = {h: i for i, h in enumerate(corpus_hashes)}
tracker = ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_FUSION)
results_table = []
print(f"Loaded {len(corpus_docs)} corpus docs.")

2026/11/27 21:14:39 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Loaded 318 corpus docs.


## 1. Extract & Cache OCR (`rapidocr_cyrillic`) + ASR (`gigaam_ctc`)

In [2]:
# Cell 2: Extract OCR & ASR once and cache to .cache/ocr_asr_318.json
if CACHE_PATH.exists():
    cached = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    ocr_texts, asr_texts = cached["ocr"], cached["asr"]
    print(f"Loaded cached OCR/ASR from {CACHE_PATH}")
else:
    ocr_ext = RapidOcrExtractor(
        min_confidence=0.5,
        rec_model_path=str(ROOT / CYRILLIC_OCR_MODEL_PATH),
        rec_keys_path=str(ROOT / CYRILLIC_OCR_DICT_PATH),
    )
    sampler = KeyframeSampler(strategy="uniform_3")
    giga_repo = "csukuangfj/sherpa-onnx-nemo-ctc-giga-am-russian-2024-10-24"
    asr_ext = SherpaOnnxAsrExtractor(
        model_type="nemo_ctc",
        nemo_ctc_path=Path(hf_hub_download(giga_repo, "model.int8.onnx")),
        tokens_path=Path(hf_hub_download(giga_repo, "tokens.txt")),
    )
    audio_ext = AudioExtractor()

    ocr_texts, asr_texts = {}, {}
    for idx, doc in enumerate(corpus_docs, 1):
        fh = doc["file_hash"]
        mpath = SILVER_DIR / doc["silver_filename"]
        if doc["media_type"] == "image":
            img = cv2.imread(str(mpath))
            ocr_texts[fh] = ocr_ext.extract_text(img).text if img is not None else ""
            asr_texts[fh] = ""
        else:
            lines = []
            for fr in sampler.sample(mpath):
                lines.extend(ocr_ext.extract_text(fr).lines)
            ocr_texts[fh] = " ".join(dict.fromkeys(lines))
            asr_texts[fh] = asr_ext.transcribe(audio_ext.extract(mpath)).text
        if idx % 50 == 0 or idx == len(corpus_docs):
            print(f"Processed {idx}/{len(corpus_docs)} items...", flush=True)

    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CACHE_PATH.write_text(json.dumps({"ocr": ocr_texts, "asr": asr_texts}, ensure_ascii=False), encoding="utf-8")
print(
    f"OCR non-empty: {sum(1 for v in ocr_texts.values() if v.strip())} | ASR non-empty: {sum(1 for v in asr_texts.values() if v.strip())}"
)

Loaded cached OCR/ASR from /home/alex/Projects/meme-organizer/.cache/ocr_asr_318.json
OCR non-empty: 209 | ASR non-empty: 93


## 2. Build 4-Channel Score Matrices (`BM25`, `Meta`, `OCR`, `ASR`)

In [3]:
# Cell 3: Load pre-trained e5-small_lora from MLflow (no retraining) and compute/cache 4-channel matrices
from meme_organizer.config import MLFLOW_EXPERIMENT_TEXT
from meme_organizer.retrieval.text.dense_text_retriever import DenseTextRetriever

splits = json.loads(SPLITS_PATH.read_text(encoding="utf-8"))
train_pairs, val_pairs, test_pairs = splits["train_pairs"], splits["val_pairs"], splits["test_pairs"]
CH_CACHE = ROOT / ".cache" / "fusion_channels_4ch.npz"
mask = MultiChannelScorer.build_channel_mask(ocr_texts, asr_texts, corpus_docs)

if CH_CACHE.exists():
    d = np.load(CH_CACHE)
    q_tr, S_tr, gt_tr = d["q_tr"], d["S_tr"], d["gt_tr"].tolist()
    q_va, S_va, gt_va = d["q_va"], d["S_va"], d["gt_va"].tolist()
    q_te, S_te, gt_te = d["q_te"], d["S_te"], d["gt_te"].tolist()
    print(f"Loaded cached 4-channel matrices from {CH_CACHE}")
else:
    e5 = DenseTextRetriever(
        "e5-small-finetuned",
        "intfloat/multilingual-e5-small",
        mlflow_experiment=MLFLOW_EXPERIMENT_TEXT,
        mlflow_run_name="e5-small_lora",
    )
    e5.load_or_build()
    docs_bm25 = [
        {
            **d,
            "rich_description": "",
            "transcript_or_text": f"{ocr_texts[d['file_hash']]} {asr_texts[d['file_hash']]}".strip(),
        }
        for d in corpus_docs
    ]
    docs_meta = [{**d, "rich_description": "", "transcript_or_text": ""} for d in corpus_docs]
    docs_ocr = [
        {**d, "user_caption": "", "tags": [], "rich_description": "", "transcript_or_text": ocr_texts[d["file_hash"]]}
        for d in corpus_docs
    ]
    docs_asr = [
        {**d, "user_caption": "", "tags": [], "rich_description": "", "transcript_or_text": asr_texts[d["file_hash"]]}
        for d in corpus_docs
    ]
    bm25 = BM25Retriever()
    bm25.load_or_build()
    c_bm25 = bm25.encode_corpus(docs_bm25)
    c_meta, c_ocr, c_asr = e5.encode_corpus(docs_meta), e5.encode_corpus(docs_ocr), e5.encode_corpus(docs_asr)

    def get_channels(pairs):
        qs = [p["query"] for p in pairs]
        gt = [hash_to_idx[p["file_hash"]] for p in pairs]
        q_e5 = e5.encode_queries(qs)
        s0 = MultiChannelScorer.min_max_normalize(bm25.predict_scores(bm25.encode_queries(qs), c_bm25))
        s1 = MultiChannelScorer.min_max_normalize(e5.predict_scores(q_e5, c_meta))
        s2 = MultiChannelScorer.min_max_normalize(e5.predict_scores(q_e5, c_ocr))
        s3 = MultiChannelScorer.min_max_normalize(e5.predict_scores(q_e5, c_asr))
        return q_e5, np.stack([s0, s1, s2, s3], axis=-1), gt

    q_tr, S_tr, gt_tr = get_channels(train_pairs)
    q_va, S_va, gt_va = get_channels(val_pairs)
    q_te, S_te, gt_te = get_channels(test_pairs)
    np.savez_compressed(
        CH_CACHE,
        q_tr=q_tr,
        S_tr=S_tr,
        gt_tr=np.array(gt_tr),
        q_va=q_va,
        S_va=S_va,
        gt_va=np.array(gt_va),
        q_te=q_te,
        S_te=S_te,
        gt_te=np.array(gt_te),
    )
print("4-channel matrices ready:", S_tr.shape, S_va.shape, S_te.shape)

Loaded cached 4-channel matrices from /home/alex/Projects/meme-organizer/.cache/fusion_channels_4ch.npz
4-channel matrices ready: (1021, 318, 4) (138, 318, 4) (153, 318, 4)


## 3. Run 1: Multi-Channel RRF (`fusion_rrf_4ch`)

In [4]:
# Cell 4: 1 MLflow run via FusionExperimentAdapter (RRF)
from meme_organizer.retrieval.fusion.fusion_experiment_adapter import FusionExperimentAdapter

with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_FUSION) as exp:
    res_rrf = exp.create_pipeline(
        FusionExperimentAdapter(
            "fusion_rrf_4ch",
            "rrf",
            q_train=q_tr,
            s_train=S_tr,
            gt_train=gt_tr,
            q_val=q_va,
            s_val=S_va,
            gt_val=gt_va,
            q_test=q_te,
            s_test=S_te,
            gt_test=gt_te,
            channel_mask=mask,
            test_queries=[p["query"] for p in test_pairs],
            corpus_docs=corpus_docs,
        )
    ).run()
results_table.append({"method": "fusion_rrf_4ch", **res_rrf["quality"], **res_rrf["performance"]})
print("RRF:", res_rrf["quality"], res_rrf["performance"])

🏃 View run fusion_rrf_4ch at: http://localhost:5000/#/experiments/13/runs/9185a9aebce94fcdb77502c90fe5aebe
🧪 View experiment at: http://localhost:5000/#/experiments/13
RRF: {'recall_at_1': 0.3007, 'recall_at_5': 0.5163, 'mrr_at_5': 0.3809, 'ndcg_at_5': 0.4146, 'total_queries': 153} {'cpu_latency_ms': 0.0517, 'cpu_latency_p95_ms': 0.0583, 'eval_inference_time_sec': 0.0087, 'eval_throughput_qps': 17521.91, 'model_size_mb': 0.0, 'total_pipeline_time_sec': 0.124}


## 4. Run 2: Optuna Weighted Fusion (`fusion_optuna_4ch`)

In [5]:
# Cell 5: 1 MLflow run via FusionExperimentAdapter (Optuna)
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_FUSION) as exp:
    res_opt = exp.create_pipeline(
        FusionExperimentAdapter(
            "fusion_optuna_4ch",
            "optuna",
            q_train=q_tr,
            s_train=S_tr,
            gt_train=gt_tr,
            q_val=q_va,
            s_val=S_va,
            gt_val=gt_va,
            q_test=q_te,
            s_test=S_te,
            gt_test=gt_te,
            channel_mask=mask,
            test_queries=[p["query"] for p in test_pairs],
            corpus_docs=corpus_docs,
        )
    ).run()
results_table.append({"method": "fusion_optuna_4ch", **res_opt["quality"], **res_opt["performance"]})
print("Optuna:", res_opt["quality"], res_opt["performance"])

🏃 View run fusion_optuna_4ch at: http://localhost:5000/#/experiments/13/runs/20d05147f7ea4199a2c8aa0951d449de
🧪 View experiment at: http://localhost:5000/#/experiments/13
Optuna: {'recall_at_1': 0.5163, 'recall_at_5': 0.6797, 'mrr_at_5': 0.5821, 'ndcg_at_5': 0.6067, 'total_queries': 153} {'cpu_latency_ms': 0.0214, 'cpu_latency_p95_ms': 0.0292, 'eval_inference_time_sec': 0.0041, 'eval_throughput_qps': 37677.45, 'model_size_mb': 0.0, 'total_pipeline_time_sec': 4.12}


## 5. Run 3: Query-Conditioned Gated Attention (`fusion_gated_attention_4ch`)

In [6]:
# Cell 6: 1 MLflow run via FusionExperimentAdapter (GatedAttention)
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_FUSION) as exp:
    res_att = exp.create_pipeline(
        FusionExperimentAdapter(
            "fusion_gated_attention_4ch",
            "gated_attention",
            q_train=q_tr,
            s_train=S_tr,
            gt_train=gt_tr,
            q_val=q_va,
            s_val=S_va,
            gt_val=gt_va,
            q_test=q_te,
            s_test=S_te,
            gt_test=gt_te,
            channel_mask=mask,
            test_queries=[p["query"] for p in test_pairs],
            corpus_docs=corpus_docs,
        )
    ).run()
results_table.append({"method": "fusion_gated_attention_4ch", **res_att["quality"], **res_att["performance"]})
print("GatedAttention:", res_att["quality"], res_att["performance"])

🏃 View run fusion_gated_attention_4ch at: http://localhost:5000/#/experiments/13/runs/548ac63822c64578a0fbbc0c7f0f5edb
🧪 View experiment at: http://localhost:5000/#/experiments/13
GatedAttention: {'recall_at_1': 0.5163, 'recall_at_5': 0.6993, 'mrr_at_5': 0.584, 'ndcg_at_5': 0.6128, 'total_queries': 153} {'cpu_latency_ms': 0.198, 'cpu_latency_p95_ms': 0.3137, 'eval_inference_time_sec': 0.0013, 'eval_throughput_qps': 116399.02, 'model_size_mb': 0.0976, 'total_pipeline_time_sec': 0.914}


## 6. Final Comparison

In [7]:
print(
    pd.DataFrame(results_table)[["method", "mrr_at_5", "recall_at_1", "recall_at_5", "ndcg_at_5"]].to_string(
        index=False
    )
)

                    method  mrr_at_5  recall_at_1  recall_at_5  ndcg_at_5
            fusion_rrf_4ch    0.3809       0.3007       0.5163     0.4146
         fusion_optuna_4ch    0.5821       0.5163       0.6797     0.6067
fusion_gated_attention_4ch    0.5840       0.5163       0.6993     0.6128
